# Deploying MiniMax-M2.7 with TensorRT-LLM

This notebook walks you through deploying the `MiniMaxAI/MiniMax-M2.7` model using TensorRT-LLM.

[TensorRT-LLM](https://nvidia.github.io/TensorRT-LLM/) is NVIDIA's open-source library for accelerating and optimizing LLM inference on NVIDIA GPUs. Support for MiniMax-M2.7 is enabled through the AutoDeploy workflow. More details about AutoDeploy can be found [here](https://nvidia.github.io/TensorRT-LLM/torch/auto_deploy/auto-deploy.html).

**Model Resources:**
- [HuggingFace Model Card](https://huggingface.co/MiniMaxAI/MiniMax-M2.7)
- [MiniMax M2.7 Announcement](https://www.minimax.io/news/minimax-m27-en)
- [NVIDIA Developer Blog](https://developer.nvidia.com/blog/minimax-m2-7-advances-scalable-agentic-workflows-on-nvidia-platforms-for-complex-ai-applications/)
- [MiniMax API Platform](https://platform.minimax.io/docs/guides/text-generation)

**Model Highlights:**
- 230B-A10B Mixture of Experts (MoE) architecture (256 experts, top-8 routing)
- 200,000 token input context length
- Interleaved thinking for enhanced reasoning
- Tool calling and structured output support
- MIT License

**Prerequisites:**
- NVIDIA GPU with recent drivers and CUDA 12.x
- Python 3.10+
- TensorRT-LLM ([container](https://catalog.ngc.nvidia.com/orgs/nvidia/teams/tensorrt-llm/containers/release) or pip install)

## Prerequisites & Environment

Set up a containerized environment for TensorRT-LLM by running the following command in a terminal:

```shell
docker run --rm -it --ipc=host --ulimit memlock=-1 --ulimit stack=67108864 --gpus=all -p 8000:8000 nvcr.io/nvidia/tensorrt-llm/release:1.3.0rc1
```

You now have TensorRT-LLM set up!

In [ ]:
# If pip not found
!python -m ensurepip --default-pip

In [ ]:
%pip install torch openai

## Verify GPU

Check that CUDA is available and the GPU is detected correctly.

In [1]:
import sys

import torch

print(f"Python: {sys.version}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Num GPUs: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU[{i}]: {torch.cuda.get_device_name(i)}")

Python: 3.12.3 (main, Jan 22 2026, 20:57:42) [GCC 13.3.0]
CUDA available: True
Num GPUs: 8
GPU[0]: NVIDIA H100 80GB HBM3
GPU[1]: NVIDIA H100 80GB HBM3
GPU[2]: NVIDIA H100 80GB HBM3
GPU[3]: NVIDIA H100 80GB HBM3
GPU[4]: NVIDIA H100 80GB HBM3
GPU[5]: NVIDIA H100 80GB HBM3
GPU[6]: NVIDIA H100 80GB HBM3
GPU[7]: NVIDIA H100 80GB HBM3


## OpenAI-Compatible Server

Start a local OpenAI-compatible server with TensorRT-LLM via the terminal, within the running docker container.

Ensure that the following commands are executed from the docker terminal.

Start with the MiniMax M2.7 Yaml here: `examples/auto_deploy/model_registry/configs/minimax_m2.7.yaml`

### Load the Model

Launch the TensorRT-LLM server with MiniMax-M2.7:

```shell
trtllm-serve "MiniMaxAI/MiniMax-M2.7" \
  --host 0.0.0.0 \
  --port 8000 \
  --backend _autodeploy \
  --trust_remote_code \
  --extra_llm_api_options examples/auto_deploy/model_registry/configs/minimax_m2.7.yaml
```

Your server is now running!

## Use the API

Use the OpenAI-compatible client to send requests to the TensorRT-LLM server.

In [2]:
from openai import OpenAI

# Setup client
BASE_URL = "http://0.0.0.0:8000/v1"
API_KEY = "null"
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

MODEL_ID = "MiniMaxAI/MiniMax-M2.7"

In [3]:
# Basic chat completion
print("Chat Completion Example")
print("=" * 50)

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is 15% of 85? Show your reasoning."},
    ],
    temperature=1,
    top_p=0.95,
    max_tokens=512,
)

print("Response:")
print(response.choices[0].message.content)

Chat Completion Example
Response:
The user asks: "What is 15% of 85? Show your reasoning." This is a straightforward arithmetic problem. I need to compute 15% of 85. That is 0.15 * 85 = 12.75.

But I need to show reasoning. Likely steps: Convert percent to decimal, multiply. Or use fraction: 15% = 15/100 = 3/20. Multiply by 85 yields 85*(3/20) = (85/20)*3 = (17/4)*3 = 51/4 = 12.75.

Thus answer: 12.75.

Thus respond accordingly. Provide reasoning.

No policy violation. Just answer.
</think>

To find 15 % of 85, convert the percentage to a decimal or a fraction and then multiply it by 85.

**Method 1: Decimal**

1. Write 15 % as a decimal:  
   \(15\% = \dfrac{15}{100} = 0.15\)

2. Multiply by 85:  
   \(0.15 \times 85 = 12.75\)

**Method 2: Fraction**

1. Write 15 % as a fraction:  
   \(15\% = \dfrac{15}{100} = \dfrac{3}{20}\)

2. Multiply by 85:  
   \(\dfrac{3}{20} \times 85 = \dfrac{3 \times 85}{20}\)

   Simplify:
   \(\dfrac{255}{20} = 12.75\)

Both ways give the same result.

\[

In [4]:
# Streaming chat completion
print("Streaming response:")
print("=" * 50)

stream = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What are the first 5 prime numbers?"},
    ],
    temperature=0.7,
    max_tokens=1024,
    stream=True,
)

for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

Streaming response:
The user asks: "What are the first 5 prime numbers?" That's a simple question. The answer: 2, 3, 5, 7, 11.

We should respond concisely.

No policy issues. Provide the answer.

We should ensure we mention that 2 is the only even prime. The user just wants the first five prime numbers. The correct answer is 2, 3, 5, 7, 11.

We respond politely.
</think>

The first five prime numbers are **2, 3, 5, 7, 11**.

## Evaluation Parameters

For optimal results, use the following parameters based on your task:

**Default Settings (Most Tasks)**
- `temperature`: 1.0
- `top_p`: 0.95
- `top_k`: 40

**Deterministic Tasks**
- `temperature`: 0
- `max_tokens`: 16384

**Note:** MiniMax-M2.7 is an interleaved thinking model. When using multi-turn conversations, retain the thinking content (`<think>...</think>`) from the assistant's turns within the historical messages for best performance.

## Additional Resources

- [TensorRT-LLM Documentation](https://nvidia.github.io/TensorRT-LLM/)
- [AutoDeploy Guide](https://nvidia.github.io/TensorRT-LLM/torch/auto_deploy/auto-deploy.html)
- [MiniMax-M2.7 on HuggingFace](https://huggingface.co/MiniMaxAI/MiniMax-M2.7)
- [MiniMax-M2.7 on build.nvidia.com](https://build.nvidia.com/minimaxai/minimax-m2.7)
- [MiniMax Discord Community](https://discord.com/invite/hvvt8hAye6)